# 11 — BTC shot coverage audit (CPU, full corpus, read-only)

Notebook này chạy **sau khi 12 worker TransNetV2 đã đủ 873/873 video**. Nó không
chạy shot detection, không tải MP4 và không cần GPU. Mục tiêu là khóa corpus rồi
đo xem từng shot đang thuộc nhóm nào:

- `C0`: không có BTC keyframe trong shot;
- `C1`: có BTC keyframe nhưng thiếu theo `K_target=ceil(duration/gap)` hoặc còn gap lớn;
- `COVERED`: đạt tiêu chí temporal đã khóa;
- `CT`: transition/micro-shot, chưa tự động bổ sung;
- `CE`: detector/audit chưa đủ tin cậy, cần review.

Kết quả này là **O1-T / temporal diagnostic**, không phải bằng chứng `NO_HEADROOM`
và không chứng minh Final Score sẽ tăng. O1-S cần feature artifacts; O1-R cần
query/rank traces. Notebook không đọc held-out, không tune threshold, không sửa
`keyframes-btc`, không sửa live store và chỉ ghi vào thư mục research riêng.

### Trước khi Save Version

1. Chọn **Accelerator: None (CPU)** và bật Internet.
2. Tạo Kaggle Secrets `GITHUB_TOKEN`, `MODAL_TOKEN_ID`, `MODAL_TOKEN_SECRET`.
3. Không chạy khi producer vẫn còn worker chưa đủ; notebook sẽ fail-closed.
4. Có thể Save Version rồi thoát. Log cuối đúng phải có `AUDIT_COMPLETE` và
   `VIDEOS: 873/873`.

## 0 — Cấu hình an toàn (thường không cần đổi)

In [ ]:
from pathlib import Path

REPO_URL = "https://github.com/AIVIETNAM-AIO-Triet-Descartes/multimodal-fusion-engine.git"
REPO_REF = "feature/quang/storage"
REPO_DIR = Path("/kaggle/working/multimodal-fusion-engine")
WORK_DIR = Path("/kaggle/working/btc-shot-coverage-audit")

EXPECTED_VIDEO_COUNT = 873
AUDIT_RUN_ID = "gate1-shot-coverage-audit-v1"
PUBLISH_RESULTS = True
CONFIRM_PUBLISH = "gate1-shot-coverage-audit-v1"
MAX_REMOTE_CONCURRENCY = 4

PRODUCER_CONFIG_REL = "config/keyframe_augmentation_kaggle.yaml"
AUDIT_CONFIG_REL = "config/keyframe_augmentation_v1.yaml"
AUDIT_SUBDIR = "gate1-audit-v1"

if PUBLISH_RESULTS and CONFIRM_PUBLISH != AUDIT_RUN_ID:
    raise RuntimeError("Publish bị khóa: CONFIRM_PUBLISH phải bằng AUDIT_RUN_ID")
if MAX_REMOTE_CONCURRENCY < 1 or MAX_REMOTE_CONCURRENCY > 5:
    raise ValueError("Giữ MAX_REMOTE_CONCURRENCY trong [1,5] theo giới hạn Modal")
WORK_DIR.mkdir(parents=True, exist_ok=True)
print({"runtime": "CPU_ONLY", "expected_videos": EXPECTED_VIDEO_COUNT, "publish": PUBLISH_RESULTS})

## 1 — Clone code đã review và ghi provenance

In [ ]:
import json
import os
import subprocess
import sys
from datetime import datetime, timezone


def run(command: list[str], *, cwd: Path | None = None, env: dict[str, str] | None = None) -> str:
    result = subprocess.run(command, cwd=cwd, env=env, check=False, text=True, capture_output=True)
    if result.returncode:
        raise RuntimeError(f"Command failed: {' '.join(command[:3])}\n{result.stderr[-1500:]}")
    return result.stdout.strip()


def clone_repo() -> None:
    if (REPO_DIR / ".git").is_dir():
        return
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("GITHUB_TOKEN")
    askpass = WORK_DIR / "git-askpass.sh"
    askpass.write_text('#!/bin/sh\ncase "$1" in *Username*) printf x-access-token ;; *) printf %s "$GITHUB_TOKEN" ;; esac\n', encoding="utf-8")
    askpass.chmod(0o700)
    env = os.environ.copy()
    env.update({"GITHUB_TOKEN": token, "GIT_ASKPASS": str(askpass), "GIT_TERMINAL_PROMPT": "0"})
    run(["git", "clone", "--depth", "1", "--branch", REPO_REF, REPO_URL, str(REPO_DIR)], env=env)
    askpass.unlink(missing_ok=True)


clone_repo()
run([sys.executable, "-m", "pip", "install", "-q", "-r", str(REPO_DIR / "notebooks/requirements-kaggle-keyframe-augmentation.txt")])
sys.path.insert(0, str(REPO_DIR / "src"))

import yaml
from mfe.offline.bundle import BundleLayout
from mfe.offline.keyframe_augmentation.common import atomic_write_json, atomic_write_jsonl, sha256_file, sha256_json
from mfe.offline.keyframe_augmentation.corpus_audit import (
    audit_video, build_summary, expected_worker_assignments, validate_corpus_index,
)
from mfe.offline.keyframe_augmentation.models import AugmentationConfig
from mfe.offline.source import load_catalogue
from mfe.offline.transport import ModalVolumeTransport

REPO_COMMIT = run(["git", "rev-parse", "HEAD"], cwd=REPO_DIR)
PRODUCER_CONFIG_PATH = REPO_DIR / PRODUCER_CONFIG_REL
AUDIT_CONFIG_PATH = REPO_DIR / AUDIT_CONFIG_REL
PRODUCER_CONFIG = yaml.safe_load(PRODUCER_CONFIG_PATH.read_text(encoding="utf-8"))
AUDIT_CONFIG = AugmentationConfig.model_validate(yaml.safe_load(AUDIT_CONFIG_PATH.read_text(encoding="utf-8")))
PRODUCER_CONFIG_SHA256 = sha256_file(PRODUCER_CONFIG_PATH)
AUDIT_CONFIG_SHA256 = sha256_file(AUDIT_CONFIG_PATH)
print({"repo_commit": REPO_COMMIT, "producer_config_sha256": PRODUCER_CONFIG_SHA256, "audit_config_sha256": AUDIT_CONFIG_SHA256})

## 2 — Kết nối hai volume và dựng inventory 12 worker (chưa ghi gì)

In [ ]:
CATALOGUE_PATH = REPO_DIR / PRODUCER_CONFIG["source"]["catalogue"]
INVENTORY = yaml.safe_load(CATALOGUE_PATH.read_text(encoding="utf-8"))
CATALOGUE = load_catalogue(CATALOGUE_PATH)
EXPECTED = expected_worker_assignments(
    producer_config=PRODUCER_CONFIG, inventory=INVENTORY, catalogue=CATALOGUE
)
EXPECTED_IDS = sorted(video_id for videos in EXPECTED.values() for video_id in videos)
if len(EXPECTED_IDS) != EXPECTED_VIDEO_COUNT or len(EXPECTED_IDS) != len(set(EXPECTED_IDS)):
    raise RuntimeError(f"Inventory không phải {EXPECTED_VIDEO_COUNT} video disjoint")

BTC = ModalVolumeTransport(PRODUCER_CONFIG["btc"]["volume"], create_if_missing=False)
RESEARCH = ModalVolumeTransport(PRODUCER_CONFIG["output"]["volume"], create_if_missing=False)
BTC_LAYOUT = BundleLayout(PRODUCER_CONFIG["btc"]["bundle"])
PRODUCER_ROOT = PRODUCER_CONFIG["output"]["root"].rstrip("/")
AUDIT_REMOTE_ROOT = f"{PRODUCER_ROOT}/{AUDIT_SUBDIR}"
if not AUDIT_REMOTE_ROOT.startswith(PRODUCER_ROOT + "/"):
    raise RuntimeError("Audit output phải nằm dưới producer research root")
print({"workers": {key: len(value) for key, value in EXPECTED.items()}, "videos": len(EXPECTED_IDS), "btc_mode": "READ_ONLY", "audit_remote_root": AUDIT_REMOTE_ROOT})

## 3 — Tải và xác minh 12 manifest + 873 COMPLETE marker

In [ ]:
from concurrent.futures import ThreadPoolExecutor, as_completed

INDEX_DIR = WORK_DIR / "index"
INDEX_DIR.mkdir(parents=True, exist_ok=True)


def fetch_json(remote: str, local: Path) -> dict:
    RESEARCH.get(remote, local)
    return json.loads(local.read_text(encoding="utf-8"))


MANIFESTS = {}
for worker in sorted(EXPECTED):
    MANIFESTS[worker] = fetch_json(
        f"{PRODUCER_ROOT}/worker-manifests/{worker}.json", INDEX_DIR / f"manifest-{worker}.json"
    )


def fetch_marker(item: tuple[str, list[str]]) -> list[tuple[str, dict]]:
    worker, video_ids = item
    rows = []
    for video_id in video_ids:
        local = INDEX_DIR / "markers" / worker / f"{video_id}.json"
        rows.append((video_id, fetch_json(f"{PRODUCER_ROOT}/markers/{MANIFESTS[worker]['member']}/{worker}/{video_id}.json", local)))
    return rows


MARKERS = {}
with ThreadPoolExecutor(max_workers=MAX_REMOTE_CONCURRENCY) as pool:
    futures = [pool.submit(fetch_marker, item) for item in sorted(EXPECTED.items())]
    for future in as_completed(futures):
        MARKERS.update(future.result())
        print({"markers_downloaded": len(MARKERS), "expected": EXPECTED_VIDEO_COUNT}, flush=True)

INDEX_REPORT = validate_corpus_index(
    producer_config=PRODUCER_CONFIG,
    expected=EXPECTED,
    manifests=MANIFESTS,
    markers=MARKERS,
    producer_config_sha256=PRODUCER_CONFIG_SHA256,
)
if INDEX_REPORT["validated_markers"] != EXPECTED_VIDEO_COUNT:
    raise RuntimeError("Corpus index chưa đủ; không audit một phần")
print("CORPUS_INDEX: PASS")
print(f"VIDEOS: {INDEX_REPORT['validated_markers']}/{EXPECTED_VIDEO_COUNT}")

## 4 — Băm lại report/shots và audit mapping BTC từng video

In [ ]:
ARTIFACT_DIR = WORK_DIR / "artifacts"
OUTPUT_DIR = WORK_DIR / "output"
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

PER_VIDEO_ROWS = []
SHOT_AUDIT_ROWS = []
REPORT_HASHES = {}
for ordinal, video_id in enumerate(EXPECTED_IDS, 1):
    marker = MARKERS[video_id]
    shots_local = ARTIFACT_DIR / "shots" / f"{video_id}.jsonl"
    report_local = ARTIFACT_DIR / "reports" / f"{video_id}.json"
    mapping_local = ARTIFACT_DIR / "mappings" / f"{video_id}.csv"
    RESEARCH.get(marker["shots"], shots_local)
    RESEARCH.get(marker["report"], report_local)
    BTC.get(BTC_LAYOUT.map_keyframes(video_id), mapping_local)
    actual_report_sha = sha256_file(report_local)
    if actual_report_sha != marker["report_sha256"]:
        raise RuntimeError(f"{video_id}: report SHA-256 mismatch")
    REPORT_HASHES[video_id] = actual_report_sha
    per_video, shot_rows = audit_video(
        video_id=video_id,
        shots_path=shots_local,
        mapping_path=mapping_local,
        marker=marker,
        config=AUDIT_CONFIG,
    )
    PER_VIDEO_ROWS.append(per_video)
    SHOT_AUDIT_ROWS.extend(shot_rows)
    if ordinal == 1 or ordinal % 25 == 0 or ordinal == EXPECTED_VIDEO_COUNT:
        print({"audited_videos": ordinal, "expected": EXPECTED_VIDEO_COUNT, "shots_so_far": len(SHOT_AUDIT_ROWS)}, flush=True)

if len(PER_VIDEO_ROWS) != EXPECTED_VIDEO_COUNT:
    raise RuntimeError("Không tạo báo cáo từ partial corpus")
SUMMARY = build_summary(per_video_rows=PER_VIDEO_ROWS, config=AUDIT_CONFIG)
SUMMARY.update({
    "audit_run_id": AUDIT_RUN_ID,
    "repo_commit": REPO_COMMIT,
    "producer_config_sha256": PRODUCER_CONFIG_SHA256,
    "audit_config_sha256": AUDIT_CONFIG_SHA256,
    "input_command": "Kaggle Save Version / Run All: notebook 11 full corpus audit",
    "heldout_accessed": False,
    "btc_mutated": False,
})

CORPUS_LOCK = {
    "schema_version": 1,
    "status": "LOCKED",
    "audit_run_id": AUDIT_RUN_ID,
    "expected_video_count": EXPECTED_VIDEO_COUNT,
    "validated_video_count": len(PER_VIDEO_ROWS),
    "worker_count": len(EXPECTED),
    "repo_commit": REPO_COMMIT,
    "producer_config_sha256": PRODUCER_CONFIG_SHA256,
    "audit_config_sha256": AUDIT_CONFIG_SHA256,
    "catalogue_sha256": sha256_file(CATALOGUE_PATH),
    "manifest_payload_hashes": INDEX_REPORT["manifest_payload_hashes"],
    "marker_payload_hashes_sha256": INDEX_REPORT["marker_payload_hashes_sha256"],
    "report_hashes_sha256": sha256_json(REPORT_HASHES),
    "per_video_input_hashes_sha256": sha256_json({row["video_id"]: {"shots": row["shots_sha256"], "mapping": row["btc_mapping_sha256"]} for row in PER_VIDEO_ROWS}),
    "detector_fingerprints": INDEX_REPORT["detector_fingerprints"],
    "btc_volume": PRODUCER_CONFIG["btc"]["volume"],
    "btc_bundle": PRODUCER_CONFIG["btc"]["bundle"],
    "btc_access": "READ_ONLY",
    "heldout_accessed": False,
}
CORPUS_LOCK["corpus_lock_hash"] = sha256_json(CORPUS_LOCK)

atomic_write_json(OUTPUT_DIR / "integrity-report.json", INDEX_REPORT)
atomic_write_json(OUTPUT_DIR / "corpus-lock.json", CORPUS_LOCK)
atomic_write_json(OUTPUT_DIR / "summary.json", SUMMARY)
atomic_write_jsonl(OUTPUT_DIR / "per-video-audit.jsonl", PER_VIDEO_ROWS)
atomic_write_jsonl(OUTPUT_DIR / "shot-audit.jsonl", SHOT_AUDIT_ROWS)
for label in ("C0", "C1", "CT", "CE"):
    atomic_write_jsonl(OUTPUT_DIR / f"{label.lower()}-shots.jsonl", (row for row in SHOT_AUDIT_ROWS if row["coverage_class"] == label))
print({"summary": SUMMARY, "corpus_lock_hash": CORPUS_LOCK["corpus_lock_hash"]})

## 5 — Publish sang research path riêng và verify (manifest ghi cuối)

In [ ]:
OUTPUT_FILES = sorted(path for path in OUTPUT_DIR.iterdir() if path.is_file())
OUTPUT_HASHES = {path.name: sha256_file(path) for path in OUTPUT_FILES}
AUDIT_MANIFEST = {
    "schema_version": 1,
    "status": "AUDIT_COMPLETE",
    "audit_run_id": AUDIT_RUN_ID,
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "repo_commit": REPO_COMMIT,
    "corpus_lock_hash": CORPUS_LOCK["corpus_lock_hash"],
    "video_count": len(PER_VIDEO_ROWS),
    "output_root": AUDIT_REMOTE_ROOT,
    "files": OUTPUT_HASHES,
    "decision_authority": "diagnostic_only",
}
AUDIT_MANIFEST["manifest_payload_hash"] = sha256_json(AUDIT_MANIFEST)
atomic_write_json(OUTPUT_DIR / "audit-manifest.json", AUDIT_MANIFEST)

if PUBLISH_RESULTS:
    for path in OUTPUT_FILES:
        remote = f"{AUDIT_REMOTE_ROOT}/{path.name}"
        RESEARCH.put(path, remote)
        if RESEARCH.sha256(remote) != OUTPUT_HASHES[path.name]:
            raise RuntimeError(f"Remote verify failed: {remote}")
    manifest_path = OUTPUT_DIR / "audit-manifest.json"
    manifest_remote = f"{AUDIT_REMOTE_ROOT}/audit-manifest.json"
    RESEARCH.put(manifest_path, manifest_remote)  # commit point: upload last
    if RESEARCH.sha256(manifest_remote) != sha256_file(manifest_path):
        raise RuntimeError("Remote audit manifest verify failed")

print("AUDIT_COMPLETE")
print(f"VIDEOS: {len(PER_VIDEO_ROWS)}/{EXPECTED_VIDEO_COUNT}")
print({"local_output": str(OUTPUT_DIR), "remote_output": AUDIT_REMOTE_ROOT if PUBLISH_RESULTS else "DISABLED", "manifest_sha256": sha256_file(OUTPUT_DIR / "audit-manifest.json")})

## 6 — Đọc kết quả và bước kế tiếp

Chỉ xem run hoàn tất khi log cuối có `AUDIT_COMPLETE`, `VIDEOS: 873/873`, và
remote manifest đã verify. Các file quan trọng:

| File | Ý nghĩa |
|---|---|
| `corpus-lock.json` | fingerprint cố định của 12 manifest, 873 marker, report, shots và mapping |
| `summary.json` | tổng C0/C1/COVERED/CT/CE; chỉ có quyền diagnostic |
| `per-video-audit.jsonl` | số liệu theo video để tìm outlier |
| `shot-audit.jsonl` | từng shot và lý do phân lớp |
| `c0-shots.jsonl`, `c1-shots.jsonl` | backlog có thể đưa vào B1–B4 sau khi review |
| `ct-shots.jsonl`, `ce-shots.jsonl` | hàng đợi human review, không tự động coi là headroom |

Tiếp theo không phải “extract hết C0/C1”. Nhóm cần review CT/CE, tạo O1-S từ
feature parity thật và O1-R từ query/rank traces, rồi mới đóng băng hard delta
budget và chạy B0–B4 cùng downstream fingerprint. Không tune bằng held-out.